In [8]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import BaseMessage

from typing import Annotated, List, Literal, Optional, Any, TypedDict, Dict
from pydantic import BaseModel, Field, ConfigDict
from datetime import datetime, date, time
from operator import add

from dotenv import load_dotenv

load_dotenv()

True

In [9]:
# ---------------------------------------------------------------------------
# Enums / Literal types
# ---------------------------------------------------------------------------

IntentType = Literal[
    "learn",
    "schedule_event",
    "add_reminder",
    "lifestyle_routine",
    "edit_schedule",
    "report_progress",
    "query_dashboard",
    "unknown",
]

GoalType = Literal["learn", "work", "routine", "social", "reminder"]

SessionType = Literal["lecture", "practice", "revision", "meeting", "routine", "reminder"]

DayOfWeek = Literal["mon", "tue", "wed", "thu", "fri", "sat", "sun"]

StreakHealth = Literal["green", "amber", "red"]

ProgressStatus = Literal["completed", "skipped", "partial", "rescheduled"]

RecoveryAction = Literal["extend_deadline", "increase_daily_hours", "drop_low_priority"]

In [10]:
# ---------------------------------------------------------------------------
# User profile: time windows, persona, preferences
# ---------------------------------------------------------------------------

class TimeWindow(BaseModel):
    """A recurring availability window on a given day of the week."""
    day: DayOfWeek
    start: time
    end: time
    label: Optional[str] = Field(default=None, description="e.g. 'morning deep work'")


class VarietyRule(BaseModel):
    """Rule used by the schedule builder to enforce session variety."""
    after_n_of_type: SessionType
    n: int = Field(ge=1, default=3)
    force_next: SessionType


class UserPreferences(BaseModel):
    timezone: str = Field(default="UTC")
    default_session_minutes: int = Field(default=90, ge=15, le=480)
    min_break_minutes: int = Field(default=15, ge=0)
    preferred_session_types: List[SessionType] = Field(default_factory=list)
    variety_rules: List[VarietyRule] = Field(default_factory=list)
    quiet_hours: List[TimeWindow] = Field(default_factory=list)


class Persona(BaseModel):
    """Soft-inferred persona — never restricts what the user can do."""
    label: Optional[str] = Field(default=None, description="e.g. 'student', 'CEO', 'retiree'")
    confidence: float = Field(default=0.0, ge=0.0, le=1.0)
    inferred_from: List[str] = Field(default_factory=list)


class UserProfile(BaseModel):
    user_id: str
    display_name: Optional[str] = None
    persona: Persona = Field(default_factory=Persona)
    time_windows: List[TimeWindow] = Field(default_factory=list)
    preferences: UserPreferences = Field(default_factory=UserPreferences)
    google_calendar_id: Optional[str] = None

In [11]:
# ---------------------------------------------------------------------------
# Goals, topics, scheduled sessions
# ---------------------------------------------------------------------------

class Topic(BaseModel):
    """One unit of curriculum produced by the topic planner."""
    topic_id: str
    title: str
    description: Optional[str] = None
    difficulty: int = Field(ge=1, le=5, description="Drives session duration in schedule builder")
    estimated_hours: float = Field(ge=0.0)
    prerequisites: List[str] = Field(default_factory=list)
    completed: bool = False
    sources: List[str] = Field(default_factory=list, description="Tavily / web URLs")


class Goal(BaseModel):
    """A user goal — learn, work, routine, social, or reminder."""
    goal_id: str
    title: str
    goal_type: GoalType
    urgency: int = Field(ge=1, le=5, default=3, description="User-declared urgency")
    deadline: Optional[date] = None
    daily_hour_budget: Optional[float] = Field(default=None, ge=0.0)
    topics: List[Topic] = Field(default_factory=list)
    progress_pct: float = Field(default=0.0, ge=0.0, le=100.0)
    priority_score: float = Field(
        default=0.0,
        description="Computed by priority manager: f(urgency, days_to_deadline, pct_remaining)",
    )
    streak_days: int = Field(default=0, ge=0)
    streak_health: StreakHealth = "green"
    created_at: datetime = Field(default_factory=datetime.utcnow)


class ScheduledSession(BaseModel):
    """One concrete event written (or to be written) to Google Calendar."""
    session_id: str
    goal_id: Optional[str] = None
    topic_id: Optional[str] = None
    title: str
    session_type: SessionType
    start: datetime
    end: datetime
    google_event_id: Optional[str] = None
    notes: Optional[str] = None
    completed: bool = False

In [12]:
# ---------------------------------------------------------------------------
# Routing, NL-edit, conflict, recovery models
# ---------------------------------------------------------------------------

class IntentClassification(BaseModel):
    intent: IntentType
    confidence: float = Field(ge=0.0, le=1.0)
    rationale: Optional[str] = None
    needs_clarification: bool = False
    clarifying_question: Optional[str] = None


class EditFilter(BaseModel):
    """Filter half of an NL edit, e.g. 'all ML sessions this week'."""
    goal: Optional[str] = None
    week: Optional[str] = Field(default=None, description="e.g. 'this_week', 'next_week', ISO week")
    day: Optional[DayOfWeek] = None
    session_type: Optional[SessionType] = None


class EditTransformation(BaseModel):
    """Transformation half of an NL edit."""
    move_to_time: Optional[time] = None
    reschedule_to_day: Optional[DayOfWeek] = None
    reduce_duration_minutes: Optional[int] = None
    cancel: bool = False


class NLEdit(BaseModel):
    filter: EditFilter
    transformation: EditTransformation
    raw_request: str


class Conflict(BaseModel):
    session_id: str
    conflicting_session_id: Optional[str] = None
    reason: str
    resolved: bool = False


class RecoveryPlan(BaseModel):
    """Surfaced by deadline guard when hours_remaining / slots_remaining > 1.0."""
    goal_id: str
    ratio: float
    options: List[RecoveryAction]
    recommended: Optional[RecoveryAction] = None
    chosen: Optional[RecoveryAction] = None


class AnalyticsEvent(BaseModel):
    """Append-only row for the dashboard aggregations."""
    event_type: Literal[
        "session_completed",
        "session_skipped",
        "goal_created",
        "goal_completed",
        "replan_triggered",
        "deadline_at_risk",
    ]
    goal_id: Optional[str] = None
    session_id: Optional[str] = None
    payload: Dict[str, Any] = Field(default_factory=dict)
    occurred_at: datetime = Field(default_factory=datetime.utcnow)

In [13]:
# ---------------------------------------------------------------------------
# Top-level LangGraph state
# ---------------------------------------------------------------------------
# Uses TypedDict with Annotated reducers so LangGraph can merge partial node
# outputs. `add_messages` handles chat history; `add` (operator.add) appends
# to log-style lists. Plain fields are overwritten on each update.

class AgentState(TypedDict, total=False):
    # ---- identity / threading -------------------------------------------------
    user_id: str
    thread_id: str
    session_run_id: str

    # ---- conversation ---------------------------------------------------------
    messages: Annotated[List[BaseMessage], add_messages]
    current_user_message: str

    # ---- routing (supervisor -> clarifier -> intent classifier) --------------
    intent: Optional[IntentClassification]
    needs_clarification: bool
    clarifying_question: Optional[str]
    next_node: Optional[str]

    # ---- user model -----------------------------------------------------------
    user_profile: Optional[UserProfile]
    user_preferences: Optional[UserPreferences]

    # ---- goals & curriculum ---------------------------------------------------
    active_goals: List[Goal]
    focus_goal_id: Optional[str]
    new_topics: List[Topic]

    # ---- schedule -------------------------------------------------------------
    proposed_sessions: List[ScheduledSession]
    scheduled_sessions: List[ScheduledSession]
    consecutive_session_type_count: Dict[str, int]

    # ---- edits ----------------------------------------------------------------
    pending_edit: Optional[NLEdit]

    # ---- progress / adaptive re-plan -----------------------------------------
    last_progress_status: Optional[ProgressStatus]
    missed_sessions: int
    replan_required: bool

    # ---- conflict resolver / deadline guard ----------------------------------
    conflicts: List[Conflict]
    deadline_at_risk: bool
    recovery_plan: Optional[RecoveryPlan]

    # ---- MCP tool layer (Calendar / Tavily / Notion / Gmail) -----------------
    pending_tool_calls: List[Dict[str, Any]]
    tool_results: List[Dict[str, Any]]

    # ---- dashboard / analytics (append-only) ---------------------------------
    analytics_events: Annotated[List[AnalyticsEvent], add]
    dashboard_snapshot: Optional[Dict[str, Any]]

    # ---- bookkeeping ----------------------------------------------------------
    errors: Annotated[List[str], add]
    updated_at: Optional[datetime]

In [14]:
# quick sanity check — the state can be instantiated and a graph can be built on it
graph = StateGraph(AgentState)



In [ ]:
# ---------------------------------------------------------------------------
# Structured-output schemas for HITL + LLM nodes
# ---------------------------------------------------------------------------
# `interrupt` and `Command` need langgraph >= 0.2.57. If the import below fails,
# upgrade with:  uv pip install -U "langgraph>=0.2.57" langgraph-checkpoint
try:
    from langgraph.types import interrupt, Command
except ImportError as e:
    raise ImportError(
        "Need langgraph>=0.2.57 for interrupt()/Command. "
        "Run: uv pip install -U \"langgraph>=0.2.57\" langgraph-checkpoint"
    ) from e

from langchain_core.messages import HumanMessage, AIMessage
import uuid


class ClarifierDecision(BaseModel):
    """Clarifier first thinks: do I have enough context to route + plan?
    If not, it returns a list of plain-string questions for the frontend
    to render Claude-style (one bubble per question)."""
    enough_context: bool = Field(
        description="True only when the agent can confidently classify intent and plan without more input.",
    )
    reasoning: str = Field(description="One-sentence justification.")
    questions: List[str] = Field(
        default_factory=list,
        description="Plain-string questions to ask the user, ordered. Empty when enough_context is True.",
    )


class TopicPlan(BaseModel):
    """Curriculum returned by topic_planner."""
    topics: List[Topic]


class ScheduleProposal(BaseModel):
    """Sessions returned by schedule_builder / lifestyle_agent."""
    sessions: List[ScheduledSession]


class NLEditExtraction(BaseModel):
    """Filter + transformation extracted from a natural-language edit."""
    filter: EditFilter
    transformation: EditTransformation

In [16]:
# ---------------------------------------------------------------------------
# LLM clients with structured output
# ---------------------------------------------------------------------------
# A single base LLM is reused; each node binds its own pydantic schema via
# .with_structured_output() so the model is forced to return a valid object.

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

clarifier_llm = llm.with_structured_output(ClarifierDecision)
intent_llm = llm.with_structured_output(IntentClassification)
topic_llm = llm.with_structured_output(TopicPlan)
schedule_llm = llm.with_structured_output(ScheduleProposal)
nl_edit_llm = llm.with_structured_output(NLEditExtraction)


CLARIFIER_SYSTEM = """You are the ChronoMind clarifier. Your job is to decide whether the agent
already has enough context from the conversation, the user's stored profile, time windows, and
active goals to plan a schedule or take an action confidently.

Rules:
- If essential information is missing (e.g. daily hour budget, deadline, prior background, preferred
  time-of-day windows, the goal itself), set enough_context=False and produce a SHORT list of
  plain, non-overlapping questions — one fact per question.
- Never bundle multiple asks into one question. The frontend renders each question as its own bubble.
- If the user message is already specific and the profile fills the rest, set enough_context=True
  and return an empty questions list.
- Do not ask about anything the user has already answered earlier in the thread."""


INTENT_SYSTEM = """Classify the user's message into one of:
learn, schedule_event, add_reminder, lifestyle_routine, edit_schedule, report_progress,
query_dashboard, unknown. Set needs_clarification=False here — clarification has already
been resolved upstream."""

NameError: name 'ClarifierDecision' is not defined

In [ ]:
# ---------------------------------------------------------------------------
# Supervisor + Clarifier (HITL #1) + Intent classifier + router
# ---------------------------------------------------------------------------

def supervisor_node(state: AgentState) -> Dict[str, Any]:
    """Pulls the latest user message and stamps run metadata. Cheap, no LLM."""
    msgs = state.get("messages", [])
    last_user = next(
        (m.content for m in reversed(msgs) if isinstance(m, HumanMessage)),
        state.get("current_user_message", ""),
    )
    return {
        "current_user_message": last_user,
        "session_run_id": state.get("session_run_id") or str(uuid.uuid4()),
        "updated_at": datetime.utcnow(),
    }


def clarifier_node(state: AgentState) -> Dict[str, Any]:
    """First decides whether the agent has enough context. If not, calls
    interrupt() with the structured questions list and waits for the
    frontend to resume with answers."""
    profile = state.get("user_profile")
    goals = state.get("active_goals", [])
    history = state.get("messages", [])

    decision: ClarifierDecision = clarifier_llm.invoke([
        {"role": "system", "content": CLARIFIER_SYSTEM},
        {"role": "user", "content": (
            f"User message: {state.get('current_user_message','')}\n"
            f"Active goals: {[g.title for g in goals]}\n"
            f"Has profile: {bool(profile)}\n"
            f"Time windows known: {bool(profile and profile.time_windows) if profile else False}\n"
            f"Recent turns: {[getattr(m,'content','')[:120] for m in history[-6:]]}"
        )},
    ])

    if decision.enough_context:
        return {"needs_clarification": False, "clarifying_question": None}

    # HITL: pause and surface the questions list to the frontend
    answer_payload = interrupt({
        "type": "clarification",
        "reasoning": decision.reasoning,
        "questions": decision.questions,
    })

    answers = (answer_payload or {}).get("answers", []) if isinstance(answer_payload, dict) else []
    qa_pairs = list(zip(decision.questions, answers))
    summary = "\n".join(f"Q: {q}\nA: {a}" for q, a in qa_pairs)

    return {
        "needs_clarification": False,
        "clarifying_question": None,
        "messages": [HumanMessage(content=f"[Clarifications]\n{summary}")],
    }


def intent_classifier_node(state: AgentState) -> Dict[str, Any]:
    classification: IntentClassification = intent_llm.invoke([
        {"role": "system", "content": INTENT_SYSTEM},
        {"role": "user", "content": state.get("current_user_message", "")},
    ])
    return {"intent": classification}


def route_by_intent(state: AgentState) -> str:
    intent = state.get("intent")
    if intent is None:
        return "mcp_tool_layer"
    mapping = {
        "learn": "topic_planner",
        "schedule_event": "schedule_builder",
        "add_reminder": "schedule_builder",
        "lifestyle_routine": "lifestyle_agent",
        "edit_schedule": "nl_edit_parser",
        "report_progress": "progress_feedback",
        "query_dashboard": "mcp_tool_layer",
    }
    return mapping.get(intent.intent, "mcp_tool_layer")

In [ ]:
# ---------------------------------------------------------------------------
# Planning nodes: topic_planner, priority_manager, schedule_builder, lifestyle_agent
# ---------------------------------------------------------------------------

TOPIC_PLANNER_SYSTEM = """You are the ChronoMind topic planner. Given a 'learn' goal, produce a
curriculum as a list of Topic objects. Assign difficulty (1-5) honestly — the schedule builder
maps difficulty to session duration. Include estimated_hours per topic. Order topics so prereqs
come first."""

SCHEDULE_BUILDER_SYSTEM = """You are the ChronoMind schedule builder. Produce ScheduledSession
objects respecting:
- the user's time windows and timezone
- session duration scaled by topic difficulty (1-2 -> 1.5h, 3 -> 2h, 4-5 -> 3h+ on weekends/long windows)
- variety rules (after N of the same session_type, force a different type)
- the daily hour budget per goal
- existing scheduled_sessions (avoid overlaps)
Use ISO datetimes. Leave google_event_id empty — the MCP layer fills it after writing to Calendar."""

LIFESTYLE_SYSTEM = """You are the ChronoMind lifestyle agent. Convert a recurring routine request
(e.g. 'every morning 5-6am meditation') into ScheduledSession objects for the next 14 days.
Use session_type='routine' (or 'reminder' for zero-duration items)."""

NL_EDIT_SYSTEM = """Extract a filter and a transformation from the user's edit request.
Filter fields: goal, week, day, session_type. Transformation fields: move_to_time,
reschedule_to_day, reduce_duration_minutes, cancel. Leave fields null when not implied."""


def topic_planner_node(state: AgentState) -> Dict[str, Any]:
    plan: TopicPlan = topic_llm.invoke([
        {"role": "system", "content": TOPIC_PLANNER_SYSTEM},
        {"role": "user", "content": (
            f"Goal: {state.get('current_user_message','')}\n"
            f"User profile timezone: {(state.get('user_profile') or UserProfile(user_id='_')).preferences.timezone}"
        )},
    ])
    # Persist topics under a fresh learn-goal if there isn't one already
    goals = list(state.get("active_goals", []))
    intent = state.get("intent")
    new_goal = Goal(
        goal_id=str(uuid.uuid4()),
        title=state.get("current_user_message", "")[:80],
        goal_type="learn",
        urgency=4 if intent and intent.confidence > 0.7 else 3,
        topics=plan.topics,
    )
    goals.append(new_goal)
    return {
        "active_goals": goals,
        "new_topics": plan.topics,
        "focus_goal_id": new_goal.goal_id,
        "analytics_events": [AnalyticsEvent(event_type="goal_created", goal_id=new_goal.goal_id)],
    }


def priority_manager_node(state: AgentState) -> Dict[str, Any]:
    """Compute priority_score = f(urgency, days_to_deadline, pct_remaining)."""
    today = date.today()
    updated: List[Goal] = []
    for g in state.get("active_goals", []):
        days = (g.deadline - today).days if g.deadline else 30
        days = max(days, 1)
        pct_remaining = max(100.0 - g.progress_pct, 1.0) / 100.0
        score = (g.urgency * 2.0) + (10.0 / days) + (pct_remaining * 5.0)
        updated.append(g.model_copy(update={"priority_score": round(score, 3)}))
    return {"active_goals": updated}


def schedule_builder_node(state: AgentState) -> Dict[str, Any]:
    goals = state.get("active_goals", [])
    profile = state.get("user_profile")
    counts = dict(state.get("consecutive_session_type_count", {}))

    proposal: ScheduleProposal = schedule_llm.invoke([
        {"role": "system", "content": SCHEDULE_BUILDER_SYSTEM},
        {"role": "user", "content": (
            f"User message: {state.get('current_user_message','')}\n"
            f"Goals (with priority_score): {[(g.title, g.priority_score, g.goal_type) for g in goals]}\n"
            f"Topics: {[(t.title, t.difficulty, t.estimated_hours) for g in goals for t in g.topics]}\n"
            f"Time windows: {[w.model_dump() for w in (profile.time_windows if profile else [])]}\n"
            f"Variety rules: {[r.model_dump() for r in (profile.preferences.variety_rules if profile else [])]}\n"
            f"Existing sessions: {len(state.get('scheduled_sessions', []))}"
        )},
    ])

    for s in proposal.sessions:
        counts[s.session_type] = counts.get(s.session_type, 0) + 1

    return {
        "proposed_sessions": proposal.sessions,
        "consecutive_session_type_count": counts,
    }


def lifestyle_agent_node(state: AgentState) -> Dict[str, Any]:
    proposal: ScheduleProposal = schedule_llm.invoke([
        {"role": "system", "content": LIFESTYLE_SYSTEM},
        {"role": "user", "content": state.get("current_user_message", "")},
    ])
    return {"proposed_sessions": proposal.sessions}


def nl_edit_parser_node(state: AgentState) -> Dict[str, Any]:
    extracted: NLEditExtraction = nl_edit_llm.invoke([
        {"role": "system", "content": NL_EDIT_SYSTEM},
        {"role": "user", "content": state.get("current_user_message", "")},
    ])
    edit = NLEdit(
        filter=extracted.filter,
        transformation=extracted.transformation,
        raw_request=state.get("current_user_message", ""),
    )

    # Apply the edit to existing scheduled_sessions to produce a proposal
    existing = state.get("scheduled_sessions", [])
    proposed: List[ScheduledSession] = []
    for s in existing:
        if extracted.filter.session_type and s.session_type != extracted.filter.session_type:
            proposed.append(s); continue
        if extracted.transformation.cancel:
            continue
        new = s
        if extracted.transformation.move_to_time:
            t = extracted.transformation.move_to_time
            new = s.model_copy(update={
                "start": s.start.replace(hour=t.hour, minute=t.minute),
                "end": s.end.replace(hour=t.hour, minute=t.minute),
            })
        if extracted.transformation.reduce_duration_minutes:
            from datetime import timedelta
            new = new.model_copy(update={
                "end": new.start + (new.end - new.start) - timedelta(minutes=extracted.transformation.reduce_duration_minutes),
            })
        proposed.append(new)
    return {"pending_edit": edit, "proposed_sessions": proposed}

In [ ]:
# ---------------------------------------------------------------------------
# Conflict resolver, deadline guard, adaptive re-planner
# ---------------------------------------------------------------------------

def _overlaps(a: ScheduledSession, b: ScheduledSession) -> bool:
    return a.start < b.end and b.start < a.end


def conflict_resolver_node(state: AgentState) -> Dict[str, Any]:
    proposed = list(state.get("proposed_sessions", []))
    existing = state.get("scheduled_sessions", [])
    conflicts: List[Conflict] = []

    # Cross-check proposed against already-scheduled
    for p in proposed:
        for e in existing:
            if _overlaps(p, e):
                conflicts.append(Conflict(
                    session_id=p.session_id,
                    conflicting_session_id=e.google_event_id or e.session_id,
                    reason=f"Overlaps existing '{e.title}' at {e.start.isoformat()}",
                ))
    # Self-overlap inside the proposal
    for i, p in enumerate(proposed):
        for q in proposed[i + 1:]:
            if _overlaps(p, q):
                conflicts.append(Conflict(
                    session_id=p.session_id,
                    conflicting_session_id=q.session_id,
                    reason="Overlaps another proposed session",
                ))
    return {"conflicts": conflicts}


def deadline_guard_node(state: AgentState) -> Dict[str, Any]:
    """ratio = hours_remaining / available_slots_before_deadline.
    If > 1.0, flip deadline_at_risk and stash a recovery plan."""
    today = date.today()
    at_risk = False
    plan: Optional[RecoveryPlan] = None

    for g in state.get("active_goals", []):
        if not g.deadline:
            continue
        days_left = max((g.deadline - today).days, 1)
        hours_remaining = sum(
            t.estimated_hours for t in g.topics if not t.completed
        )
        budget = g.daily_hour_budget or 2.0
        slots = days_left * budget
        ratio = hours_remaining / max(slots, 0.1)
        if ratio > 1.0:
            at_risk = True
            plan = RecoveryPlan(
                goal_id=g.goal_id,
                ratio=round(ratio, 3),
                options=["extend_deadline", "increase_daily_hours", "drop_low_priority"],
                recommended="increase_daily_hours" if ratio < 1.5 else "extend_deadline",
            )
            break

    out: Dict[str, Any] = {"deadline_at_risk": at_risk, "recovery_plan": plan}
    if at_risk:
        out["analytics_events"] = [AnalyticsEvent(
            event_type="deadline_at_risk",
            goal_id=plan.goal_id if plan else None,
            payload={"ratio": plan.ratio if plan else None},
        )]
    return out


def adaptive_replanner_node(state: AgentState) -> Dict[str, Any]:
    """Triggered when missed_sessions >= 2 OR deadline_at_risk is True.
    Asks the schedule LLM to redistribute remaining hours across remaining days."""
    goals = state.get("active_goals", [])
    proposal: ScheduleProposal = schedule_llm.invoke([
        {"role": "system", "content": SCHEDULE_BUILDER_SYSTEM + "\nRedistribute backlog across remaining days."},
        {"role": "user", "content": (
            f"missed_sessions={state.get('missed_sessions', 0)}; "
            f"deadline_at_risk={state.get('deadline_at_risk', False)}; "
            f"goals={[(g.title, g.priority_score, g.deadline.isoformat() if g.deadline else None) for g in goals]}"
        )},
    ])
    return {
        "proposed_sessions": proposal.sessions,
        "replan_required": False,
        "analytics_events": [AnalyticsEvent(event_type="replan_triggered")],
    }

In [ ]:
# ---------------------------------------------------------------------------
# HITL #2: approval gate -> MCP tool layer -> progress feedback
# ---------------------------------------------------------------------------

def approval_gate_node(state: AgentState) -> Dict[str, Any]:
    """Hard pause before any Calendar mutation. Surfaces the proposal to the
    user; only when resume payload contains approved=True do we let the MCP
    layer write to Google Calendar."""
    proposed = state.get("proposed_sessions", [])
    if not proposed:
        return {"pending_tool_calls": []}

    decision = interrupt({
        "type": "approval",
        "summary": f"{len(proposed)} session(s) ready to schedule",
        "conflicts": [c.model_dump() for c in state.get("conflicts", [])],
        "deadline_at_risk": state.get("deadline_at_risk", False),
        "recovery_plan": (state.get("recovery_plan").model_dump() if state.get("recovery_plan") else None),
        "sessions": [s.model_dump(mode="json") for s in proposed],
    })

    approved = bool((decision or {}).get("approved")) if isinstance(decision, dict) else bool(decision)

    if not approved:
        return {
            "pending_tool_calls": [],
            "proposed_sessions": [],
            "messages": [AIMessage(content="Schedule discarded — nothing was written to your calendar.")],
        }

    profile = state.get("user_profile")
    cal_id = profile.google_calendar_id if profile else "primary"
    calls: List[Dict[str, Any]] = [
        {
            "tool": "google_calendar.create_event",
            "args": {
                "calendar_id": cal_id,
                "summary": s.title,
                "start": s.start.isoformat(),
                "end": s.end.isoformat(),
                "description": s.notes or "",
                "session_id": s.session_id,
            },
        }
        for s in proposed
    ]
    return {"pending_tool_calls": calls}


def mcp_tool_layer_node(state: AgentState) -> Dict[str, Any]:
    """Calls the MCP server (FastMCP) for Calendar/Tavily/Notion/Gmail. Stubbed
    here — replace with a real MCP client. After a successful create_event we
    move proposed_sessions into scheduled_sessions and append analytics."""
    calls = state.get("pending_tool_calls", [])
    if not calls:
        return {}

    results: List[Dict[str, Any]] = []
    moved: List[ScheduledSession] = list(state.get("scheduled_sessions", []))
    proposed = {s.session_id: s for s in state.get("proposed_sessions", [])}
    new_analytics: List[AnalyticsEvent] = []

    for call in calls:
        if call["tool"] == "google_calendar.create_event":
            sid = call["args"]["session_id"]
            fake_event_id = f"gcal_{uuid.uuid4().hex[:10]}"
            results.append({"tool": call["tool"], "ok": True, "google_event_id": fake_event_id})
            if sid in proposed:
                moved.append(proposed[sid].model_copy(update={"google_event_id": fake_event_id}))
                new_analytics.append(AnalyticsEvent(
                    event_type="session_completed",
                    session_id=sid,
                    payload={"phase": "scheduled"},
                ))

    return {
        "tool_results": results,
        "scheduled_sessions": moved,
        "proposed_sessions": [],
        "pending_tool_calls": [],
        "analytics_events": new_analytics,
        "messages": [AIMessage(content=f"Scheduled {len(new_analytics)} session(s) to your calendar.")],
    }


def progress_feedback_node(state: AgentState) -> Dict[str, Any]:
    """Handles 'I finished today's session' / 'I skipped'. Increments
    missed_sessions on a skip — when it hits 2 we set replan_required so the
    next turn routes through adaptive_replanner."""
    msg = state.get("current_user_message", "").lower()
    if any(k in msg for k in ("skip", "missed", "couldn't", "didn't")):
        status: ProgressStatus = "skipped"
        missed = state.get("missed_sessions", 0) + 1
        replan = missed >= 2
    elif any(k in msg for k in ("done", "finished", "completed")):
        status = "completed"
        missed = state.get("missed_sessions", 0)
        replan = False
    else:
        status = "partial"
        missed = state.get("missed_sessions", 0)
        replan = False

    return {
        "last_progress_status": status,
        "missed_sessions": missed,
        "replan_required": replan,
        "analytics_events": [AnalyticsEvent(
            event_type="session_completed" if status == "completed" else "session_skipped",
            payload={"status": status},
        )],
        "messages": [AIMessage(content=f"Logged: {status}. (missed_sessions={missed})")],
    }

In [ ]:
# ---------------------------------------------------------------------------
# Build the graph — InMemorySaver + HITL interrupts at clarifier and approval
# ---------------------------------------------------------------------------
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(AgentState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("clarifier", clarifier_node)
builder.add_node("intent_classifier", intent_classifier_node)
builder.add_node("topic_planner", topic_planner_node)
builder.add_node("schedule_builder", schedule_builder_node)
builder.add_node("priority_manager", priority_manager_node)
builder.add_node("lifestyle_agent", lifestyle_agent_node)
builder.add_node("nl_edit_parser", nl_edit_parser_node)
builder.add_node("conflict_resolver", conflict_resolver_node)
builder.add_node("deadline_guard", deadline_guard_node)
builder.add_node("adaptive_replanner", adaptive_replanner_node)
builder.add_node("approval_gate", approval_gate_node)
builder.add_node("mcp_tool_layer", mcp_tool_layer_node)
builder.add_node("progress_feedback", progress_feedback_node)

# Linear pre-amble: supervisor -> clarifier -> intent_classifier
builder.add_edge(START, "supervisor")
builder.add_edge("supervisor", "clarifier")
builder.add_edge("clarifier", "intent_classifier")

# Intent fan-out
builder.add_conditional_edges(
    "intent_classifier",
    route_by_intent,
    {
        "topic_planner": "topic_planner",
        "schedule_builder": "schedule_builder",
        "lifestyle_agent": "lifestyle_agent",
        "nl_edit_parser": "nl_edit_parser",
        "progress_feedback": "progress_feedback",
        "priority_manager": "priority_manager",
        "mcp_tool_layer": "mcp_tool_layer",
    },
)

# Topic planner always feeds priority manager then schedule builder
builder.add_edge("topic_planner", "priority_manager")
builder.add_edge("priority_manager", "schedule_builder")

# Anything that produces sessions goes through conflict resolver -> deadline guard
builder.add_edge("schedule_builder", "conflict_resolver")
builder.add_edge("lifestyle_agent", "conflict_resolver")
builder.add_edge("nl_edit_parser", "conflict_resolver")
builder.add_edge("conflict_resolver", "deadline_guard")

# Deadline guard branches: at risk -> replanner; otherwise -> approval gate
builder.add_conditional_edges(
    "deadline_guard",
    lambda s: "adaptive_replanner" if s.get("deadline_at_risk") or s.get("replan_required") else "approval_gate",
    {"adaptive_replanner": "adaptive_replanner", "approval_gate": "approval_gate"},
)
builder.add_edge("adaptive_replanner", "approval_gate")

# HITL approval gate -> MCP tool layer -> END
builder.add_edge("approval_gate", "mcp_tool_layer")
builder.add_edge("mcp_tool_layer", END)
builder.add_edge("progress_feedback", END)

checkpointer = InMemorySaver()
app = builder.compile(checkpointer=checkpointer)

# Render Mermaid for inspection
try:
    print(app.get_graph().draw_mermaid())
except Exception as e:
    print("Mermaid render skipped:", e)

In [ ]:
# ---------------------------------------------------------------------------
# Example run — three turns: initial -> answer clarifications -> approve schedule
# ---------------------------------------------------------------------------
import uuid
from langchain_core.messages import HumanMessage

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

initial_state: AgentState = {
    "user_id": "demo-user",
    "thread_id": thread_id,
    "session_run_id": str(uuid.uuid4()),
    "current_user_message": "I want to learn machine learning in 30 days",
    "messages": [HumanMessage(content="I want to learn machine learning in 30 days")],
    "active_goals": [],
    "scheduled_sessions": [],
    "proposed_sessions": [],
    "consecutive_session_type_count": {},
    "missed_sessions": 0,
    "replan_required": False,
    "deadline_at_risk": False,
    "conflicts": [],
    "pending_tool_calls": [],
    "tool_results": [],
    "analytics_events": [],
    "errors": [],
    "needs_clarification": False,
}

# Turn 1 — graph runs until it hits the clarifier interrupt with questions
turn1 = app.invoke(initial_state, config=config)
print("Turn 1 interrupt payload:")
print(turn1.get("__interrupt__"))

# Turn 2 — frontend collected answers; resume
turn2 = app.invoke(
    Command(resume={"answers": [
        "About 2 hours per weekday and 4 hours on weekends",
        "I have no prior ML background",
        "Evenings 7-9pm work best",
    ]}),
    config=config,
)
print("\nTurn 2 interrupt payload (approval gate):")
print(turn2.get("__interrupt__"))

# Turn 3 — user approves the proposed schedule
turn3 = app.invoke(Command(resume={"approved": True}), config=config)
print("\nFinal scheduled sessions:", len(turn3.get("scheduled_sessions", [])))
print("Analytics events:", [e.event_type for e in turn3.get("analytics_events", [])])